In [ ]:
# ============================================================
# IMPORTS AND CONSTANTS
# ============================================================
# Standard numerical/plotting libraries
# mumax3PP: custom post-processing library for mumax3 micromagnetic simulation output
#   - ovf:    reads .ovf vector-field files (magnetization snapshots)
#   - parameters: constructs OVF file naming/metadata
#   - fft_across_xyzm: performs FFT over time axis across all spatial points and components
# Physical constants defined as short aliases for convenience
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
%matplotlib inline
import multiprocessing as mp
import matplotlib.colors as colors
import scipy.optimize as opt
from scipy.signal import find_peaks
import mumax3PP.ovf as ovf
import mumax3PP.parameters as parameters
import mumax3PP.fft_across_xyzm as FFT_across_xyzm
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
# import mumax3PP.plotsRGB as pltRGB
import imageio

import glob
import os.path
import time
from os import path
from IPython.display import clear_output

import time

plt.rcParams.update({'font.size': 11})


sqrt = np.sqrt
pi = np.pi
exp = np.exp
sin = np.sin
cos = np.cos
mu0 = pi*4e-7

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================
# findNearest(array, value):
#   Returns the index and value of the element in  closest to .
#   Used throughout to locate specific frequencies, positions, etc.
#
# gedDT(times):
#   Estimates the time step dt from a (possibly irregular) time array
#   by fitting a linear model t = a*i + b and returning the slope a.
#   More robust than simple differencing when timestamps have small jitter.
#
# kalDR(k, angle_z, angle_k, d, mu0H0eff, wM, lb, mu0Ha, gamma):
#   Kalinikos-Slavin dispersion relation for a magnetic thin film.
#   Computes the spin-wave frequency f(k) for a given in-plane wavevector k.
#   Parameters:
#     k         - wavevector magnitude (rad/m)
#     angle_z   - polar angle of magnetization w.r.t. film normal (rad)
#     angle_k   - azimuthal angle of k-vector w.r.t. magnetization (rad)
#     d         - film thickness (m)
#     mu0H0eff  - effective DC field mu0*H_eff (T)
#     wM        - characteristic frequency gamma*mu0*Ms (rad/s)
#     lb        - exchange stiffness parameter (m^2 rad/s), = mu0*Ms*l_ex^2
#     mu0Ha     - uniaxial anisotropy field mu0*H_a (T)
#     gamma     - gyromagnetic ratio (rad/s/T)
#   Returns frequency in Hz.
def findNearest(array,value):
    idx = (np.abs(array-value)).argmin()
    return idx, array[idx]

def gedDT(times):
    xdata = range(len(times))
    from scipy.optimize import curve_fit
    def func(x, a, b):
        return a * x + b
    dt=np.abs(times[-1]-times[1])/(len(times)-1)
    popt, pcov = curve_fit(func, xdata, times, p0=[dt,0])
#     plt.plot(xdata, times)
#     plt.plot(xdata, func(xdata, *popt))
    return popt[0]

def kalDR(k, angle_z, angle_k, d, mu0H0eff, wM, lb, mu0Ha, gamma):
#   kalDR(k, angle_z*(pi/180), angle_k*(pi/180), d, mu0Heff, wM, lb, mu0Ha, gamma)

    def Fcoef1(k, theta, phi, w0):
        P = 1 - (1-exp(-k*d))/(k*d)
        return P+(sin(theta)**2) *(1-P*(1+cos(phi)**2) +  sin(phi)**2*(wM/(w0)) * (1-exp(-2*k*d))/4 )
    
    w0 = gamma*mu0H0eff +  gamma*lb*k**2
    w2 = w0*(w0 + wM*Fcoef1(k, angle_z, angle_k, w0) )
    return np.sqrt(w2)/(2*np.pi)

In [ ]:
# ============================================================
# SIMULATION SAMPLING PARAMETERS
# ============================================================
# Define the excitation bandwidth and temporal sampling for the spin-wave simulation.
# f_cut:     cutoff (maximum) frequency of the sinc excitation pulse (Hz)
# T:         period corresponding to f_cut
# t_sampl:   time step satisfying Nyquist criterion with 1.5x oversampling margin
# n_samples: total number of time steps for ~1000 excitation periods
# df:        frequency resolution of the resulting FFT
# f_max:     maximum frequency resolvable (= 1/(2*t_sampl), Nyquist limit)
f_cut = 12e09
T = 1/f_cut

t_sampl = 0.5/(f_cut*1.5)     #T/10
n_samples = int((1000*T)/t_sampl)

df = 1/(t_sampl*n_samples)
f_max = 1/(2*t_sampl)

print("Max frequency: {} GHz".format(  round(f_max*1e-9,2)   ))    
print("time sampling: {} ps".format(t_sampl*1e12))
print("freq sampling: {} GHz".format(df*1e-9))
print("Samples: {}".format( int(n_samples) ))

In [ ]:
# ============================================================
# SINC EXCITATION PULSE
# ============================================================
# Construct a sinc pulse centered at t0 = 500*t_sampl.
# sinc(f_cut * (t - t0)) excites all frequencies uniformly from 0 to f_cut,
# making it the standard broadband excitation used in spin-wave spectroscopy simulations.
# The plot shows the time-domain waveform in nanoseconds.
t = np.arange(0.1, n_samples, 1)*t_sampl
t0 = 500*t_sampl
signal = np.sin(  2*np.pi*f_cut*(t-t0)   )/(2*np.pi*f_cut*(t-t0))

plt.plot(t*1e9, signal  )
plt.xlabel("t (ns)")
plt.show()

In [ ]:
t0

In [ ]:
# ============================================================
# FFT OF EXCITATION PULSE (VERIFICATION)
# ============================================================
# Verify the flat spectrum of the sinc pulse up to f_cut.
# Only the positive-frequency half of the FFT is plotted (sh = N//2).
# The red dashed vertical line marks f_cut to confirm the spectral cutoff.
sp = np.fft.fft(signal)
freq = np.fft.fftfreq(t.shape[0],t_sampl)
sh = sp.shape[0]//2

dfi = freq[1] - freq[0]
# print(dfi/1e9)

plt.plot(freq[:sh]*1e-9, np.abs(sp)[:sh]  )
plt.ylim(0, np.abs(sp)[:sh].max()*1.025)
plt.xlabel("f (GHz)")
plt.axvline(x=f_cut/1e9, c="r",lw=0.75,ls="--")
plt.show()

In [ ]:
# ============================================================
# MATERIAL PARAMETERS (PRELIMINARY)
# ============================================================
# Compute the exchange length l_ex = sqrt(2A / mu0*Ms^2) for reference.
# These values (Ms=16 kA/m, A=1.37 pJ/m) appear to be initial estimates;
# the main simulation uses updated parameters defined later in Cell 10.
Ms = 16e3  
A  = 1.37e-12
l_ex = np.sqrt(2*A/(mu0*Ms**2))

print( "Exchange length: {} nm".format(  round(l_ex*1e9,2)   )  )
print( "Saturation field: {} mT".format(   round((Ms*mu0)*1000 ),2)   )  

In [ ]:
# ============================================================
# LOAD STATIC (EQUILIBRIUM) FIELD PROFILE
# ============================================================
# Read the spatially non-uniform static magnetic field produced by the
# superconducting structure from a mumax3 .ovf file.
# The field has three vector components (Bx, By, Bz) on an (x, y) grid.
# x, y arrays are constructed from the cell size and centered at zero.
#
# The applied bias field value is parsed from the filename (e.g. B_150.0mT).
# Vertical dashed lines mark the edges of the 800 nm-wide region.
# The plot shows how the stray field from the superconductor modifies
# the local effective field experienced by the magnetic film.
dir0 = r"C:\Users\admin\Documents\mumax3\superconductor\spectra"
head = "static_field_width_800nm_B_150.0mT"
parms = parameters.ovfParms(head=head, )
# dir0 = r"C:\Users\admin\Documents\mumax3\superconductor\spectra\800nm"
# parms = parameters.ovfParms(head="init2", )
M_tzyxm = ovf.OvfFile(dir0, parms)
print((M_tzyxm.array).shape)

B_stat = M_tzyxm.array[:,:,:,:,:]

cx = M_tzyxm._headers["xstepsize"]
x = np.arange(0, M_tzyxm.array.shape[3]*cx-cx/10,cx)
x -= x.max()/2

cy = M_tzyxm._headers["ystepsize"]
y = np.arange(0, M_tzyxm.array.shape[2]*cy-cy/10,cy)
y -= y.max()/2


plt.figure(figsize=(10,6))
plt.plot(x*1e6, B_stat[0,0,0,:,0], lw=3, label=r"$B_x$")
plt.plot(x*1e6, B_stat[0,0,0,:,1], lw=3, label=r"$B_y$")
plt.plot(x*1e6, B_stat[0,0,0,:,2]+float(head.split("_")[-1][:-2])*1e-3, lw=3, label=r"$B_z$")

plt.axvline(x=-0.4,c="k")
plt.axvline(x= 0.4,c="k")

plt.xlim(-2,2)

plt.xlabel(r"x ($\mathrm{\mu}$m)")
plt.ylabel(r"Change in external magnetic field ($\Delta$T)")
plt.ylabel(r"External magnetic field (T)")
plt.legend()
# plt.savefig(  r"C:\Users\admin\Documents\mumax3\superconductor\spectra\field.png",bbox_inches='tight',pad_inches = 0,dpi=600,facecolor='w', )
plt.show()

In [ ]:
# ============================================================
# MATERIAL PARAMETERS (MAIN)
# ============================================================
# Define the magnetic material parameters for the GaYIG (gallium yttrium iron garnet) film.
# Ms:     saturation magnetization derived from mu0*Ms = 20.2 mT
# B_ani:  uniaxial anisotropy field (94.1 mT)
# Ku1:    uniaxial anisotropy energy density = 0.5 * B_ani * Ms
# A:      exchange stiffness constant (1.37 pJ/m)
# d:      film thickness (20 nm)
# gamma:  gyromagnetic ratio (179 GHz/T)
# wM:     characteristic magnon frequency gamma*mu0*Ms
# mu0Ha:  anisotropy field 2*Ku/Ms
# lex2:   square of exchange length = 2A/(mu0*Ms^2)
# lb:     exchange stiffness in dispersion units = mu0*Ms*lex2
# angle_k/angle_z: spin-wave propagation geometry (Damon-Eshbach configuration, 90deg)
Ms = 20.2e-3/mu0
print("Ms = " + str(round(Ms/1e3,2)) + " (kA/m)"  )

B_ani = (94.1e-3)
Ku1 = 0.5*B_ani*Ms

mu0 = pi*4e-07
d = 20e-09 #thickness
# Ms = 16074
A = 1.37e-12
print("A_ex = " + str(round(A/1e-12,2)) + " (pJ/m)"  )
print("Ku1 = " + str(round(Ku1,2)) + " (J/m3)"  )

gamma = 179e09
wM = gamma*mu0*Ms
Ku = Ku1  #756.3122473198412
mu0Ha = 2*Ku/Ms
lex2 = 2*A/(mu0*Ms**2)
lb =  mu0*Ms*lex2
# B0 =  float(dir0.split("_")[-2][:-2])*1e-3

print("  ")
print("exchange length, l_ex= " + str(  round(np.sqrt(lex2)*1e9,2)  ) + " (nm)"  )

# k = np.linspace(-100.1e6, 100e6, int(1e03))
angle_k = 90*(np.pi/180)
angle_z = 0*(np.pi/180)

In [ ]:
# ============================================================
# DATA PATH CONFIGURATION
# ============================================================
# Path to the mumax3 simulation output directory for a specific run.
# Filename encodes the geometry (400nm constriction width) and bias field (90 mT).
# The path is parsed later to extract these parameters automatically.
dir0 = r"C:\Users\admin\Documents\mumax3\superconductor\spectra\400nm\GaYIG_B0_90.0mT_field_profile_Bdyn_x_spectrum_fcut_12GHz.out"


In [ ]:
# ============================================================
# LOAD DYNAMIC SIMULATION DATA
# ============================================================
# Load three datasets from the simulation output directory:
#   1. B_stat: static (equilibrium) field profile under the superconductor
#   2. M1:     time-resolved magnetization m(t, x) — the main dynamic dataset
#              Shape: (n_times, nz, ny, nx, 3) — 3 vector components
#   3. B1:     time-resolved dynamic magnetic field B(t, x) (starting from frame 2)
# x, y coordinate arrays are built from the grid cell sizes and centered at zero.
path0 = r"C:\Users\admin\Documents\mumax3\superconductor\spectra"
parms = parameters.ovfParms(head="static_field_width_{}_B_{}".format(dir0.split("\\")[-2],
                                                                            dir0.split("\\")[-1].split("_")[2]), )
B0 = float(dir0.split("\\")[-1].split("_")[2][:-2])*1e-3


M_tzyxm = ovf.OvfFile(path0, parms)
print((M_tzyxm.array).shape)
B_stat = M_tzyxm.array[:,:,:,:,:]

parms = parameters.ovfParms(head="m", )
M_tzyxm = ovf.OvfFile(dir0, parms)
print((M_tzyxm.array).shape)

M1 = M_tzyxm.array[:,:,:,:,:]
t1 = M_tzyxm.time

cx = M_tzyxm._headers["xstepsize"]
x = np.arange(0, M_tzyxm.array.shape[3]*cx-cx/10,cx)
x -= x.max()/2

cy = M_tzyxm._headers["ystepsize"]
y = np.arange(0, M_tzyxm.array.shape[2]*cy-cy/10,cy)
y -= y.max()/2

parms = parameters.ovfParms(head="B", tStart=2)
B_tzyxm = ovf.OvfFile(dir0, parms)
print((B_tzyxm.array).shape)
B1 = B_tzyxm.array[:,:,:,:,:]
t2 = B_tzyxm.time

# parms = parameters.ovfParms(head="init")
# M_tzyxm = ovf.OvfFile(dir0, parms)
# print((M_tzyxm.array).shape)
# M0 = M_tzyxm.array[:,:,:,:,:]

In [ ]:
# ============================================================
# FERROMAGNETIC RESONANCE (FMR) FREQUENCY
# ============================================================
# Compute the uniform-mode FMR frequency using the Kalinikos-Slavin dispersion.
# mu0Heff is the effective field at the film center:
#   H_eff = B0 (applied) - mu0*Ms (demagnetization) + mu0*Ha (anisotropy)
# The dispersion is evaluated at k ~ 0 (wavevector 1e-6 rad/m ≈ 0)
# to get the spatially-uniform precession frequency fq.
mu0Heff = B0 - mu0*Ms + mu0Ha

fq = kalDR(1e-6, angle_z*(pi/180), angle_k*(pi/180), d, mu0Heff, wM, lb, mu0Ha, gamma)

In [ ]:
# ============================================================
# LOW-PASS SPATIAL FILTER ON MAGNETIZATION
# ============================================================
# Apply a k-space low-pass filter to each magnetization component independently.
# For each time step, FFT along x, zero out all wavevectors |k| > 2e8 rad/m,
# then IFFT back. This suppresses high-spatial-frequency numerical noise
# while retaining physically relevant spin-wave modes.
# The filter is applied to all three components: mx, my, mz.
signal = M1[:,0,0,:,0]
sp = np.fft.fft(signal)
k_vec = np.fft.fftfreq(x.shape[0],cx)
mask = np.abs(k_vec) < 2e8
sp = sp*mask 
sh = sp.shape[0]//2
M1[:,0,0,:,0] = np.real(np.fft.ifft(sp))

signal = M1[:,0,0,:,1]
sp = np.fft.fft(signal)
k_vec = np.fft.fftfreq(x.shape[0],cx)
mask = np.abs(k_vec) < 2e8
sp = sp*mask 
sh = sp.shape[0]//2
M1[:,0,0,:,1] = np.real(np.fft.ifft(sp))

signal = M1[:,0,0,:,2]
sp = np.fft.fft(signal)
k_vec = np.fft.fftfreq(x.shape[0],cx)
mask = np.abs(k_vec) < 2e8
sp = sp*mask 
sh = sp.shape[0]//2
M1[:,0,0,:,2] = np.real(np.fft.ifft(sp))

In [ ]:
# ============================================================
# DYNAMIC MAGNETIZATION AMPLITUDES
# ============================================================
# Extract the dynamic (AC) part of each magnetization component by
# subtracting the static equilibrium value (first time frame).
#   amp_x/y/z = m(t, x) - m(t=0, x)
# r = |delta_m| is the total oscillation amplitude (all components).
# The time trace is plotted at the position closest to x = -400 nm
# (edge of the superconducting structure), showing the local spin-wave amplitude vs time.
comp = 0
amp_x =  M1[:,0,0,:,comp]  -  M1[0,0,0,:,comp] #  
comp = 1
amp_y =  M1[:,0,0,:,comp]  -  M1[0,0,0,:,comp] #  
comp = 2
amp_z =  M1[:,0,0,:,comp]  -  M1[0,0,0,:,comp] # 

r = np.sqrt( amp_x**2 + amp_y**2 + amp_z**2 )    

# for i in range(  250, r.shape[0] ):
#     clear_output(wait=True)
#     plt.title("{}/{}".format(i+1, np.shape(r)[0]))  
#     plt.plot(x,  amp_x[i], label=r"$m_{x}$")
#     plt.plot(x,  amp_y[i], label=r"$m_{y}$")
# #     plt.plot(x,  amp_z[i], label=r"$m_{y}$")
#     plt.ylim( -r.max()*0.99, r.max()*0.99 )
#     plt.axvline(x=-400e-9, c="k", ls="--", lw=0.75)
#     plt.axvline(x= 400e-9, c="k", ls="--", lw=0.75)
#     plt.legend()
#     plt.show()

plt.plot(t1*1e9, r[:,  findNearest(x,-400e-9)[0]  ]     )
plt.show()

In [ ]:
# ============================================================
# DYNAMIC FIELD ANALYSIS AT A FIXED POINT
# ============================================================
# Analyse the y-component of the dynamic field B_y at a fixed spatial index (x index 2048).
# Two plots are produced:
#   1. Time-domain signal B_y(t) at the selected point
#   2. FFT spectrum |B_y(f)| showing which frequencies are excited at that location
# This confirms that the excitation spectrum covers the intended frequency range.
comp = 0
B_x =  B1[:,0,0,:,comp] 
comp = 1
B_y =  B1[:,0,0,:,comp] 
comp = 2
B_z =  B1[:,0,0,:,comp] 

A = np.abs(B_y).max() #0.01*0.09

# for i in range(   B_y.shape[0]  ):

#     clear_output(wait=True)
#     plt.title("{}/{}".format(i+1, np.shape(B_y)[0]))  
# #     plt.plot(x,  B_x[i], label=r"$B_{x}$")
#     plt.plot(x,  B_y[i], label=r"$B_{y}$")
# #     plt.plot(x,  B_z[i], label=r"$B_{z}$")
#     plt.ylim(-A, A)
#     plt.axvline(x=-400e-9, c="k", ls="--", lw=0.75)
#     plt.axvline(x= 400e-9, c="k", ls="--", lw=0.75)
#     plt.legend()
#     plt.show()

plt.plot( t2*1e9,   B_y[:,2048]  )
# plt.ylim(-0.005*B0, 0.025*B0)
plt.xlim(0, 1.01*t1.max()*1e9)
plt.show()

sp = np.fft.fft( B_y[:,2048] )
freq = np.fft.fftfreq(t2.shape[0],  t2[1]-t2[0]  )
sh = sp.shape[0]//2

plt.plot(freq[1:sh]*1e-9, np.abs(sp)[1:sh]  )
plt.ylim(0, np.abs(sp)[1:sh].max()*1.025)
plt.xlim(freq[1:sh].min()*1e-9, freq[1:sh].max()*1e-9)
plt.xlabel("f (GHz)")
# plt.axvline(x=f_cut/1e9, c="r",lw=0.75,ls="--")
plt.show()

In [ ]:
# ============================================================
# SPATIOTEMPORAL FFT — SPIN-WAVE SPECTRUM
# ============================================================
# Perform the full 4D FFT of the magnetization data M1 across time and all spatial axes
# using the custom FFT_across_xyzm routine.
# MM[f, nz, ny, nx, comp]: complex Fourier amplitudes at each frequency and position
# fqs: corresponding frequency array
#
# fmr: spatially-averaged spectral amplitude |MM(f)| — gives the global spin-wave spectrum.
# f0i: index of the dominant (FMR) peak in the spectrum.
# f0:  frequency of the dominant peak (printed in GHz).
comp = 0

MM, fqs = FFT_across_xyzm.FFT_across_xyzm(M1[:,:,:,:,:], t1[:-1])
fmr = np.mean(np.abs(MM[:,:,:,:,comp]), axis=(1,2,3))
f0i = findNearest(fmr, fmr[1:].max())[0]
M1_f0 = MM[f0i,0,:,:,0]/M1.shape[0]/0.5
f0 = fqs[f0i]
print("f_peak  = {} GHz".format( f0/1e9  ) )
print("df = {} GHz".format( fqs[1]/1e9 ) )

In [ ]:
# ============================================================
# SPIN-WAVE SPECTRUM PLOT WITH PEAK DETECTION
# ============================================================
# Plot the spatially-averaged spin-wave spectrum fmr(f) and annotate peaks.
# find_peaks locates resonance peaks above 1% of the maximum amplitude.
# The magenta dashed line marks the analytically predicted FMR frequency fq.
# Black dashed lines mark all detected peaks BELOW the FMR frequency —
# these sub-FMR peaks correspond to localised spin-wave modes confined
# in the potential well created by the non-uniform field under the superconductor.
# The x-axis is limited to the excitation bandwidth (f_cut from the filename).
peaks, _ = find_peaks(fmr, height=fmr[1:].max()*0.01)

plt.rcParams.update({'font.size': 12})


plt.figure(figsize=(10,6))
plt.title(dir0.split("\\")[-1][:-4])
plt.plot(fqs[1:]*1e-9,fmr[1:])   #  ,label="FFT")
plt.xlabel(r"f (GHz)")
plt.ylabel(r"FFT amp (a.u.)")

plt.axvline(x=fq/1e9, c="magenta", ls="--", label="FMR, {} GHz".format( round(fq/1e9,2) )  )
# plt.axvline(x=fq.min()/1e9, c="lime", ls="--", label="Well Bottom, {} GHz".format( round(fq[0]/1e9,2) )  )

nn = 1
for i in range( np.shape(peaks)[0] ): 
    if fqs[peaks[i]]*1e-9 > 2 and fqs[peaks[i]]*1e-9 < 1.*fq/1e9:
        plt.axvline( fqs[peaks[i]]*1e-9, c="k", ls="--", lw=0.5,
                   label = "Peak {}. = ".format(nn) + "{} GHz".format( round(fqs[peaks[i]]*1e-9,2 )  ) )
        nn += 1



plt.xlim(0, float(dir0.split("_")[-1][:-7])   )
plt.ylim(0, fmr[1:].max()*1.025   )
plt.legend(loc="upper right")
# plt.savefig(  r"C:\Users\admin\Documents\mumax3\superconductor\spectra\\" + dir0.split("\\")[-2] + "_" 
#             +  dir0.split("\\")[-1][:-4]+f".png",bbox_inches='tight',pad_inches = 0,dpi=600,facecolor='w', )
plt.show()

In [ ]:
# ============================================================
# MODE PROFILES OF SUB-FMR PEAKS
# ============================================================
# For each detected peak below the FMR frequency, plot the spatial mode profile.
# Each subplot shows:
#   Left y-axis (blue): |m_y(x)| at the peak frequency — the spin-wave amplitude profile
#   Right y-axis (black): static field B_z(x) + B0 — the non-uniform potential well
# Vertical dashed lines mark the physical edges of the superconducting structure.
# This reveals which modes are spatially confined (localised) within the field minimum.
for i in range(peaks.shape[0] ):

    
    f0 = peaks[i]
    freq = round(fqs[f0]*1e-9,2)
    M1_f0 = MM[f0,0,:,:,comp]/M1.shape[0]/0.5

    if fqs[peaks[i]]*1e-9 > 2 and fqs[peaks[i]]*1e-9 < 1.*fq/1e9:

        print(i)

        fig, ax1 = plt.subplots(figsize=(10,6))
        plt.title("w = {}, B0 = {}, f={} GHz".format( dir0.split("\\")[-2], dir0.split("\\")[-1].split("_")[2],freq))
        color = 'tab:red'
        ax1.set_xlabel(r"x ($\mathrm{\mu}$m)")
        ax1.set_ylabel(r"amp (a.u.)", color='tab:blue')
        ax1.plot( x*1e6,  np.abs(M1_f0[0,:]), label=r"Re[$m_y$]"  )
        ax1.axhline(y=0,c="tab:blue",ls="--",lw=0.5)
#         ax1.set_ylim( -0.1e-7, 4e-5)
        # ax1.tick_params(axis='y', labelcolor=color)

        ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

        ax2.set_ylabel(r"$B_\mathrm{ext}$ (T)", color='k')  # we already handled the x-label with ax1
        ax2.plot(x*1e6, B_stat[0,0,0,:,2]+B0,
                         label=r"$B_z$",    c="k"    )
        ax2.axvline(x=-1*float(dir0.split("\\")[-2][:-2])*1e-3/2,c="k",ls="--",lw=0.5)
        ax2.axvline(x=float(dir0.split("\\")[-2][:-2])*1e-3/2,c="k",ls="--",lw=0.5)
        ax2.set_ylim(0, 0.24)
        plt.xlim( -3., 3.  )
        fig.tight_layout()  # otherwise the right y-label is slightly clipped

    #         plt.savefig(r"C:\Users\admin\Documents\mumax3\superconductor\spectra\figs\\" + dir0.split("\\")[-2] + "_" 
    #                         +dir0.split("\\")[-1][:-4]+r"_mode_f_{}GHz.jpg".format(freq),bbox_inches='tight',pad_inches = 0,dpi=600)
        plt.show()

In [ ]:
# ============================================================
# EXPORT A SINGLE MODE PROFILE TO TEXT FILE
# ============================================================
# Extract and save the spatial amplitude profile of a specific mode (peak index 47)
# to a text file for external analysis or plotting.
# Columns: x position (metres) and complex amplitude m_y(x).
# Filename encodes the geometry, simulation label, and peak frequency.
f0 = peaks[47]
freq = round(fqs[f0]*1e-9,2)
M1_f0 = MM[f0,0,:,:,comp]/M1.shape[0]/0.5
print(i)

plt.figure(figsize=(10,6))
plt.title("f={} GHz".format(  freq  ))
plt.plot( x*1e6,  np.abs(M1_f0[0,:]), label=r"Re[$m_y$]"  )
plt.show()


file1 = open(r'C:\Users\admin\superconductor\modes\{}_{}_{}.txt'.format(dir0.split("\\")[-2], 
                                                                     dir0.split("\\")[-1][:-4], freq), 'w+')
file1.write('x (mum)    amp (a.u.)\n'  )

for i in range(x.shape[0]):
    file1.write('{}\t'.format(x[i])  )
    file1.write('{}\n'.format(M1_f0[0,i])  )

file1.close()